# PySpark Pandas I/O

The **pandas API on Spark** supports reading and writing data using familiar
pandas-style I/O methods backed by Spark's distributed datasources.

This notebook covers:
1. CSV — read & write
2. Parquet — read & write
3. JSON — read & write
4. Spark DataFrame I/O interop
5. ORC and other Spark datasources

## Setup

In [2]:
import os
import tempfile
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'
os.environ['JAVA_HOME'] = os.environ['JAVA_HOME_11']
import numpy as np
import pandas as pd
import pyspark.pandas as ps
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('pyspark-pandas-io')
    .master(os.environ.get('SPARK_MASTER', 'local[*]'))
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

OUTPUT_DIR = os.environ.get('OUTPUT_PATH', tempfile.mkdtemp(prefix='spp_io_'))
print(f'Spark {spark.version} — output: {OUTPUT_DIR}')

KeyError: 'JAVA_HOME_11'

## Sample Data

In [ ]:
psdf = ps.DataFrame({
    'id':      [1, 2, 3, 4, 5],
    'name':    ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'score':   [85.5, 92.0, 78.0, 88.5, 95.0],
    'city':    ['NYC', 'LA', 'NYC', 'LA', 'NYC'],
})
psdf

## 1. CSV

Write and read CSV files using `ps.DataFrame.to_csv()` and `ps.read_csv()`.

In [ ]:
csv_path = os.path.join(OUTPUT_DIR, 'people.csv')
psdf.to_csv(csv_path, index=False)
print(f'Written to {csv_path}')

In [ ]:
psdf_csv = ps.read_csv(csv_path)
psdf_csv

## 2. Parquet

Parquet is the preferred format — columnar, compressed, and schema-aware.

In [ ]:
parquet_path = os.path.join(OUTPUT_DIR, 'people.parquet')
psdf.to_parquet(parquet_path)
print(f'Written to {parquet_path}')

In [ ]:
psdf_parquet = ps.read_parquet(parquet_path)
psdf_parquet

## 3. JSON

JSON is useful for semi-structured data and API interop.

In [ ]:
json_path = os.path.join(OUTPUT_DIR, 'people.json')
psdf.to_json(json_path)
print(f'Written to {json_path}')

In [ ]:
psdf_json = ps.read_json(json_path)
psdf_json

## 4. Spark DataFrame I/O

Convert to a Spark DataFrame to use Spark's native readers/writers,
then convert back.

In [ ]:
# pandas-on-Spark → Spark → write partitioned Parquet
partitioned_path = os.path.join(OUTPUT_DIR, 'people_by_city')
sdf = psdf.to_spark()
sdf.write.mode('overwrite').partitionBy('city').parquet(partitioned_path)
print(f'Partitioned parquet written to {partitioned_path}')

In [ ]:
# Read back via Spark and convert to pandas-on-Spark
sdf_back = spark.read.parquet(partitioned_path)
sdf_back.printSchema()

psdf_back = sdf_back.pandas_api()
psdf_back.sort_values('id')

## 5. ORC & Other Datasources

The pandas API on Spark supports any Spark datasource via the Spark
DataFrame bridge. Use `to_spark()` and the native reader/writer.

In [ ]:
orc_path = os.path.join(OUTPUT_DIR, 'people.orc')
psdf.to_spark().write.mode('overwrite').orc(orc_path)
print(f'ORC written to {orc_path}')

psdf_orc = spark.read.orc(orc_path).pandas_api()
psdf_orc

## Cleanup

In [ ]:
import shutil
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
print(f'Cleaned up {OUTPUT_DIR}')
spark.stop()